In [1]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
import kagglehub
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertTokenizer, BertModel

# ==========================================
# 1. DEVICE SETUP (Colab GPU Check)
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {device}")

# ==========================================
# 2. LOAD CACHED KAGGLEHUB DATASET (70-15-15)
# ==========================================
print("📂 Fetching local dataset path from kagglehub cache...")
# Automatically retrieves the absolute file path from your downloaded cache
cache_dir = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
csv_path = os.path.join(cache_dir, "IMDB Dataset.csv")

df = pd.read_csv(csv_path)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Perfect 70-15-15 Split from exactly 50,000 labeled rows
train_texts, remaining_texts, train_labels, remaining_labels = train_test_split(
    df['review'].tolist(), df['label'].tolist(), test_size=0.30, random_state=42, stratify=df['label'].tolist()
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    remaining_texts, remaining_labels, test_size=0.50, random_state=42, stratify=remaining_labels
)

print(f"📦 Total Labeled Data: {len(df)}")
print(f"📊 Dataset Splits -> Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

# ==========================================
# 3. TOKENIZATION (BERT-Base)
# ==========================================
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==========================================
# 4. MODEL ARCHITECTURE (Frozen BERT + Custom NN)
# ==========================================
class BERT_Arch(nn.Module):
    def __init__(self, bert_model_name):
        super(BERT_Arch, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)

        # Lock/Freeze base BERT layers
        for param in self.bert.parameters():
            param.requires_grad = False

        self.fc1 = nn.Linear(768, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(256, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_hidden_state = outputs.pooler_output
        x = self.fc1(cls_hidden_state)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)
        return logits

model = BERT_Arch(model_name).to(device)

# ==========================================
# 5. LOSS AND OPTIMIZER
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

# Exact layout format matching Hugging Face flat-bar screenshot style
PBAR_FORMAT = "{desc}: {percentage:3.0f}% |{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]"
EPOCHS = 5

# ==========================================
# 6. TRAINING LOOP (Flat Green Style)
# ==========================================
print(f"\n🟢 Training custom layers for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        unit="batch",
        bar_format=PBAR_FORMAT,
        colour="#388e3c"  # Exact Hugging Face download-green color
    )

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    print(f"✨ Epoch {epoch+1} Done | Average Loss: {total_loss/len(train_loader):.4f}\n")

# ==========================================
# 7. VALIDATION LOOP
# ==========================================
print("🔍 Running Validation...")
model.eval()
val_preds = []
val_labels_all = []

val_pbar = tqdm(
    val_loader,
    desc="Evaluating",
    unit="batch",
    bar_format=PBAR_FORMAT,
    colour="#388e3c"
)

with torch.no_grad():
    for batch in val_pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        val_preds.extend(preds.cpu().numpy())
        val_labels_all.extend(labels.cpu().numpy())

print(f"\n🎯 Validation Accuracy: {accuracy_score(val_labels_all, val_preds):.4f}")

# ==========================================
# 8. FINAL TESTING LOOP (Holdout Set)
# ==========================================
print("\n🏁 Evaluating Unseen Test Set...")
test_preds = []
test_labels_all = []

test_pbar = tqdm(
    test_loader,
    desc="Testing",
    unit="batch",
    bar_format=PBAR_FORMAT,
    colour="#388e3c"
)

with torch.no_grad():
    for batch in test_pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        test_preds.extend(preds.cpu().numpy())
        test_labels_all.extend(labels.cpu().numpy())

print(f"\n🔥 Final Holdout Test Accuracy: {accuracy_score(test_labels_all, test_preds):.4f}")
print("\n📊 Full Classification Report:")
print(classification_report(test_labels_all, test_preds, target_names=['Negative', 'Positive']))


🚀 Using device: cuda
📂 Fetching local dataset path from kagglehub cache...
Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
📦 Total Labeled Data: 50000
📊 Dataset Splits -> Train: 35000 | Val: 7500 | Test: 7500


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

✨ Epoch 5 Done | Average Loss: 0.5114

🔍 Running Validation...


Evaluating: 100% |██████████| 235/235 [00:48<00:00,  4.80batch/s]



🎯 Validation Accuracy: 0.7699

🏁 Evaluating Unseen Test Set...


Testing: 100% |██████████| 235/235 [00:48<00:00,  4.81batch/s]


🔥 Final Holdout Test Accuracy: 0.7775

📊 Full Classification Report:
              precision    recall  f1-score   support

    Negative       0.82      0.71      0.76      3750
    Positive       0.74      0.85      0.79      3750

    accuracy                           0.78      7500
   macro avg       0.78      0.78      0.78      7500
weighted avg       0.78      0.78      0.78      7500

